# depth-pose-boxing — 3D Skeleton Visualisation

Inspect the normalised (T, 9, 3) skeleton output from `main.py`.

**What this notebook shows:**
1. 3D skeleton for a single frame — all 9 joints with bones
2. Wrist trajectories over time — punch motion in X, Y, Z
3. Joint position table — raw XYZ numbers for the first 5 frames

In [ ]:
import sys
sys.path.append('..')  # allow imports from project root

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from src.constants import (
    ACTIVE_JOINT_NAMES,
    SKELETON_EDGES,
    BOXING_JOINTS,
    NUM_ACTIVE_JOINTS,
)

# ── Load your .npy file ───────────────────────────────────────────────
# Change this path to your processed output
NPY_PATH = '../data/processed/punch_iphone13.npy'

sequence = np.load(NPY_PATH)   # (T, 9, 3)
T = sequence.shape[0]

print(f'Loaded: {NPY_PATH}')
print(f'Shape : {sequence.shape}  →  {T} frames, {NUM_ACTIVE_JOINTS} joints, 3 axes (X Y Z)')
print(f'X range: [{sequence[:,:,0].min():.3f}, {sequence[:,:,0].max():.3f}]')
print(f'Y range: [{sequence[:,:,1].min():.3f}, {sequence[:,:,1].max():.3f}]')
print(f'Z range: [{sequence[:,:,2].min():.3f}, {sequence[:,:,2].max():.3f}]')

## Cell 1 — 3D Skeleton (single frame)

All 9 joints plotted in 3D space. Rotate interactively with your mouse.

- 🔴 Red — wrists (punch endpoints)
- 🔵 Blue — shoulders
- 🟢 Green — hips
- ⚪ White — nose

In [ ]:
# Change this to inspect a different frame
FRAME_IDX = 0

# Colour map — local joint index → colour
JOINT_COLOURS = {
    0: 'white',    # nose
    1: '#4A90D9',  # left_shoulder  — blue
    2: '#4A90D9',  # right_shoulder — blue
    3: '#A0C4FF',  # left_elbow     — light blue
    4: '#A0C4FF',  # right_elbow    — light blue
    5: '#FF4B4B',  # left_wrist     — red
    6: '#FF4B4B',  # right_wrist    — red
    7: '#51CF66',  # left_hip       — green
    8: '#51CF66',  # right_hip      — green
}

frame = sequence[FRAME_IDX]   # (9, 3)

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

# Draw bones
for (i, j) in SKELETON_EDGES:
    xs = [frame[i, 0], frame[j, 0]]
    ys = [frame[i, 1], frame[j, 1]]
    zs = [frame[i, 2], frame[j, 2]]
    ax.plot(xs, zs, ys, color='#555577', linewidth=2, alpha=0.8)

# Draw joints
for j in range(NUM_ACTIVE_JOINTS):
    x, y, z = frame[j]
    colour   = JOINT_COLOURS[j]
    size     = 120 if j in [5, 6] else 80  # wrists larger
    ax.scatter(x, z, y, c=colour, s=size, zorder=5,
               edgecolors='white', linewidths=0.5)
    ax.text(x, z, y + 0.05,
            ACTIVE_JOINT_NAMES[j].replace('_', '\n'),
            fontsize=6, color='#cccccc', ha='center')

ax.set_xlabel('X', color='#aaaaaa')
ax.set_ylabel('Z (depth)', color='#aaaaaa')
ax.set_zlabel('Y', color='#aaaaaa')
ax.tick_params(colors='#aaaaaa')
ax.set_title(f'3D Skeleton — Frame {FRAME_IDX}', color='white', pad=15)

plt.tight_layout()
plt.show()

## Cell 2 — Wrist Trajectories Over Time

X, Y, Z position of both wrists across all frames.
Punch delivery shows as a sharp spike in X or Z.

In [ ]:
# Local joint indices
L_WRIST = 5
R_WRIST = 6
L_SHOULDER = 1
R_SHOULDER = 2

frames = np.arange(T)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.patch.set_facecolor('#1a1a2e')

axis_labels = ['X  (left / right)', 'Y  (up / down)', 'Z  (depth)']
colours = {
    'left_wrist':    '#FF4B4B',
    'right_wrist':   '#FF9F43',
    'left_shoulder': '#4A90D9',
    'right_shoulder':'#A0C4FF',
}

for dim, (ax, label) in enumerate(zip(axes, axis_labels)):
    ax.set_facecolor('#12122a')

    ax.plot(frames, sequence[:, L_WRIST,    dim],
            color=colours['left_wrist'],    linewidth=1.5, label='Left wrist')
    ax.plot(frames, sequence[:, R_WRIST,    dim],
            color=colours['right_wrist'],   linewidth=1.5, label='Right wrist')
    ax.plot(frames, sequence[:, L_SHOULDER, dim],
            color=colours['left_shoulder'], linewidth=1,
            linestyle='--', alpha=0.6,      label='Left shoulder')
    ax.plot(frames, sequence[:, R_SHOULDER, dim],
            color=colours['right_shoulder'],linewidth=1,
            linestyle='--', alpha=0.6,      label='Right shoulder')

    ax.set_ylabel(label, color='#aaaaaa')
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333355')
    ax.axhline(0, color='#444466', linewidth=0.8, linestyle=':')
    ax.legend(loc='upper right', fontsize=8,
              facecolor='#1a1a2e', labelcolor='white', framealpha=0.7)

axes[-1].set_xlabel('Frame', color='#aaaaaa')
fig.suptitle('Wrist & Shoulder Trajectories — X / Y / Z',
             color='white', fontsize=13, y=1.01)

plt.tight_layout()
plt.show()

## Cell 3 — Joint Position Table

Raw XYZ coordinates for the first 5 frames.
Confirms the numbers are real and the pipeline ran correctly.

In [ ]:
N_FRAMES = 5   # change to inspect more frames

# Header
col_w  = 10
header = f"{'Joint':<18}" + "".join(
    [f"{'Frame '+str(i):>{col_w*3}}" for i in range(N_FRAMES)]
)
subhdr = f"{'':18}" + "".join(
    [f"{'X':>{col_w}}{'Y':>{col_w}}{'Z':>{col_w}}" for _ in range(N_FRAMES)]
)

print(header)
print(subhdr)
print('-' * (18 + col_w * 3 * N_FRAMES))

for j in range(NUM_ACTIVE_JOINTS):
    name    = ACTIVE_JOINT_NAMES[j]
    marker  = ' ◀' if j in [5, 6] else ''   # highlight wrists
    row     = f"{name + marker:<18}"
    for t in range(N_FRAMES):
        x, y, z = sequence[t, j]
        row += f"{x:>{col_w}.3f}{y:>{col_w}.3f}{z:>{col_w}.3f}"
    print(row)

print('-' * (18 + col_w * 3 * N_FRAMES))
print(f'\nTotal frames in sequence: {T}')
print(f'Output shape: {sequence.shape}  →  (T, 9 joints, 3 axes)')